# INFO
* 자기장센서 계측 Log를 통해, classification
* initialValueLog: 초기 Offset 값 계측 로그
* logData: 자석이 위치한 상태에서의 계측 로그


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyts.image import GramianAngularField
from keras.models import load_model

In [2]:
########## Global Variables ##########
sensorIdx = [4, 9, 14, 20, 25, 30]
sensorName = [f"s{i}" for i in np.arange(1, 7, 1)]
gaf = GramianAngularField(method= "difference")
colName = "col"
skipNumber = 300
useRow = 4000

In [3]:
initialValueLog = pd.read_table("../data/initialValue.log", names= [colName])
initialValues = []
for i in range(initialValueLog.shape[0]):
    if len(initialValueLog.iloc[i, 0]) > 41 or len(initialValueLog.iloc[i, 0]) < 39:
        pass
    else:
        initialValues.append(initialValueLog.iloc[i, 0])
initialValues = np.array(initialValues)

initialSensors = []
for i in sensorIdx:
    initialSensors.append([s[i : i + 5] for s in initialValues])
initialSensors = pd.DataFrame(np.array(initialSensors, dtype= np.float32).T, columns= sensorName)

# meanInitialSensors = pd.DataFrame(initialSensors.mean(axis= 0), columns= ["init"])
meanInitialSensors = initialSensors.mean(axis= 0).to_numpy()

In [4]:
logData = pd.read_table("../data/class21.log", names= [colName], skiprows= skipNumber, nrows= useRow)
data = []
for i in range(logData.shape[0]):
    if len(logData.iloc[i, 0]) > 41 or len(logData.iloc[i, 0]) < 39:
        pass
    else:
        data.append(logData.iloc[i, 0])
data = np.array(data)

sensorData = []
for i in sensorIdx:
    sensorData.append([d[i : i + 5] for d in data])
sensorData = np.array(sensorData, dtype= np.float32).T
sensorData.shape

(4000, 6)

In [5]:
calibrated = [sensorData[:, i] - meanInitialSensors[i] for i in range(6)]
calibrated = np.array(calibrated).reshape(-1, 6)
calibrated.shape

(4000, 6)

In [6]:
splited = []
for i in range(int(calibrated.shape[0] / 16)):
    splited.append(calibrated[i * 16 : 16 + (i * 16), :])
splited = np.array(splited)
splited.shape

(250, 16, 6)

In [7]:
encode = []
for i in range(splited.shape[0]):
    res = []
    for j in range(splited.shape[2]):
        s = splited[i, :, j].reshape(-1, 1)
        e = gaf.fit_transform(s.T).reshape(1, 16, 16)
        res.append(e)
    res = np.array(res)
    encode.append(res)
encode = np.array(encode)
encode.shape

(250, 6, 1, 16, 16)

In [8]:
reshaped = encode.reshape(250, 1, 16, 16, 6)

In [9]:
trainModel = load_model("../data/Test0919_6Channel_Epoch_100.h5")

predictResult = []
for i in range(reshaped.shape[0]):
    predictClass = np.argmax(trainModel.predict(reshaped[i]), axis= 1)
    predictResult.append(predictClass)

Metal device set to: Apple M2 Pro

systemMemory: 32.00 GB
maxCacheSize: 10.67 GB



2025-03-18 18:14:40.304223: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-03-18 18:14:40.304337: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-03-18 18:14:41.663002: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2025-03-18 18:14:41.877108: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


1/1 [==============================] - 0s 17ms/step


In [10]:
print(np.unique(predictResult, return_counts= True))

(array([ 9, 11, 15, 21, 26]), array([  1,   3,   2, 124, 120]))
